# Task 2: multi-head self-attention with model-wide variables

**Step 1: before anything else, type your participant number (written at the top of your task sheet, digits only) in the `participant_id` field below and run the cell** (click ▶ on its left). It starts the timer of this task.

**Step 2:** open *Your model* (click ▸ next to the title) and implement the model of **Task 2** on the task sheet. *Try it* runs your model on a batch. No training is needed.

**Step 3:** when you are done, open *Submit*. You submit once: the answer is final.

In [ ]:
#@title ▶ Step 1: run this cell first (starts the task timer)
participant_id = ""  #@param {type:"string"}

import math
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

import importlib

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False
# Study helpers: in Google Colab always the current ones from GitHub, else from the repository.
if IN_COLAB:
    import urllib.request

    RAW = "https://raw.githubusercontent.com/LucaSforza/nnmodelling-ui/expert-study-materials/"
    urllib.request.urlretrieve(RAW + "tools/expert_study/study_checks.py", "study_checks.py")
    urllib.request.urlretrieve(RAW + "examples/test/datasets/mnist/data.json", "mnist_small.json")
    sys.path.insert(0, os.getcwd())
    importlib.invalidate_caches()
for rel in ("../../../tools/expert_study", "../../../../tools/expert_study"):
    if os.path.isdir(rel):
        sys.path.append(os.path.abspath(rel))
import study_checks

study_checks.begin("T2", participant_id)
x, y = study_checks.load_batch()
print("Task T2 started. x", tuple(x.shape), x.dtype, "| y", tuple(y.shape), y.dtype)


## Step 2: Your model

Implement `MultiHeadRowAttentionClassifier` as in **Task 2** on the task sheet. `d_model` and `num_heads` play the role of the model-wide variables: your submission is checked with `(d_model, num_heads)` = (32, 4), (64, 8) and (32, 2), including the attention scale `1 / sqrt(d_model / num_heads)` in every head.

In [ ]:
# Allowed: nn.Linear, nn.ReLU / torch.relu, nn.Flatten / torch.flatten, @ / torch.matmul,
#   transpose, scalar multiplication, softmax, torch.cat, +, nn.ModuleList, Python loops,
#   view / reshape (to split heads), F.cross_entropy / nn.CrossEntropyLoss.
# Forbidden: nn.MultiheadAttention, F.scaled_dot_product_attention,
#   F.multi_head_attention_forward, nn.Transformer*.

class Head(nn.Module):
    def __init__(self, d_model, d_head):
        super().__init__()
        self.q = nn.Linear(d_model, d_head)
        self.k = nn.Linear(d_model, d_head)
        self.v = nn.Linear(d_model, d_head)
        self.scale = 1 / math.sqrt(d_head)

    def forward(self, h):
        scores = self.q(h) @ self.k(h).transpose(-2, -1) * self.scale
        return torch.softmax(scores, dim=-1) @ self.v(h)


class MultiHeadRowAttentionClassifier(nn.Module):
    def __init__(self, d_model=32, num_heads=4):
        super().__init__()
        self.embed = nn.Linear(28, d_model)
        self.classifier = nn.Linear(28 * d_model, 10)
        self.heads = nn.ModuleList(Head(d_model, d_model // num_heads) for _ in range(num_heads))

    def forward(self, x):
        h = self.embed(torch.flatten(x, 1, 2))
        attn = torch.cat([head(h) for head in self.heads], dim=-1)
        h = h + attn
        return self.classifier(torch.flatten(h, 1))


In [ ]:
def compute_loss(logits, target):
    return F.cross_entropy(logits, target)  # provided


## Try it (optional)

Runs your model on the batch and prints shapes, loss and parameter count. It does not check your answer.

In [ ]:
# Optional, run it as often as you like: shapes, loss and parameter count of your model on the
# batch (what NNModelling shows in the Inspector and in Model > Parameter counts).
# It does not check your answer.
model = MultiHeadRowAttentionClassifier(d_model=32, num_heads=4)  # try other values too
logits = model(x)
loss = compute_loss(logits, y)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("logits", tuple(logits.shape), "| loss", round(loss.item(), 4), "| trainable parameters", params)


## ⚠️ Step 3: Submit — remove the `#` in the cell below and run it ONCE

- Your answer is **final**: only the first submission counts, running it again changes nothing.
- No result is shown. A file `nnm-study-T2-….json` is saved to your **Downloads** folder: allow the download if the browser asks and leave the file there.
- Then tell the facilitator.

In [ ]:
# Remove the "#" at the start of the next line, then run this cell ONCE.
# study_checks.submit("T2", MultiHeadRowAttentionClassifier, compute_loss)
